In [1]:
import os
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
PREP = os.path.join(REPO_ROOT, "data", "2.preprocessing")

# [Cell 1] 토크나이저의 토큰을 길이 순서대로 나열
from transformers import AutoTokenizer

# 확인할 토크나이저 경로 (BERT_Scratch 도메인 특화 토크나이저)
TOKENIZER_PATH = os.path.join(PREP, "2.scratch", "tokenizer")

print(f"'{TOKENIZER_PATH}' 토크나이저 로딩 중...")
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH)

# 토크나이저의 전체 vocab 가져오기 ({'토큰': 인덱스} 형태의 딕셔너리)
vocab = tokenizer.get_vocab()

# 서브워드 접두사(## 등)를 제외한 실제 글자 길이를 기준으로 정렬하기 위한 전처리
# 예: '##아세트아미노펜' -> '아세트아미노펜' (길이 7)
token_lengths = []
for token in vocab.keys():
    clean_token = token.replace("##", "")
    token_lengths.append((token, len(clean_token)))

# 길이 내림차순으로 정렬
sorted_tokens = sorted(token_lengths, key=lambda x: x[1], reverse=True)

print("\n[토큰 길이 순 정렬 결과 (상위 50개)]")
print(f"{'순위':>4s} | {'토큰':>25s} | {'길이':>4s}")
print("-" * 40)

for i, (token, length) in enumerate(sorted_tokens[:50], 1):
    print(f"{i:>4d} | {token:>25s} | {length:>4d}")

'./saved/3.scratch_v2/tokenizer' 토크나이저 로딩 중...

[토큰 길이 순 정렬 결과 (상위 50개)]
  순위 |                        토큰 |   길이
----------------------------------------
   1 |        lipopolysaccharide |   18
   2 |        methylprednisolone |   18
   3 |         metalloproteinase |   17
   4 |          aminotransferase |   16
   5 |          sulfamethoxazole |   16
   6 |          cyclophosphamide |   16
   7 |           characteristics |   15
   8 |         ##opolysaccharide |   15
   9 |            대한뇌졸중학회지제6권제1호 |   14
  10 |            clarithromycin |   14
  11 |            bisphosphonate |   14
  12 |            chromatography |   14
  13 |            metalloprotein |   14
  14 |            picrylhydrazyl |   14
  15 |            classification |   14
  16 |            aminoglycoside |   14
  17 |            characteristic |   14
  18 |            대한악안면성형재건외과학회지 |   14
  19 |            Staphylococcus |   14
  20 |             metronidazole |   13
  21 |             dehydrogenase |   13
  22 |

In [2]:
import os
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
PREP = os.path.join(REPO_ROOT, "data", "2.preprocessing")

# [Cell 2] 사용자 입력을 받아 두 토크나이저의 분절 결과 비교
from transformers import AutoTokenizer

BASELINE_PATH = os.path.join(PREP, "1.dap", "tokenizer")
SCRATCH_PATH = os.path.join(PREP, "2.scratch", "tokenizer")

print("비교용 토크나이저 로딩 중...")
try:
    baseline_tok = AutoTokenizer.from_pretrained(BASELINE_PATH)
    scratch_tok = AutoTokenizer.from_pretrained(SCRATCH_PATH)
    print("토크나이저 로드 완료.\n")
except Exception as e:
    print(f"토크나이저 로드 실패: {e}")

print("=" * 60)
print("토크나이저 분절 비교 테스트 (종료하려면 'q' 또는 'quit' 입력)")
print("=" * 60)

while True:
    user_input = input("\n분절할 텍스트를 입력하세요: ").strip()
    
    if user_input.lower() in ['q', 'quit']:
        print("테스트를 종료합니다.")
        break
        
    if not user_input:
        continue

    # 토크나이징 수행
    base_tokens = baseline_tok.tokenize(user_input)
    scratch_tokens = scratch_tok.tokenize(user_input)
    
    # 토큰 개수 계산
    base_len = len(base_tokens)
    scratch_len = len(scratch_tokens)
    
    print(f"\n[입력 문장] {user_input}")
    print("-" * 60)
    print(f"▶ Baseline (토큰 수: {base_len})")
    print(f"  {' '.join(base_tokens)}")
    print("-" * 60)
    print(f"▶ Scratch_v2 (토큰 수: {scratch_len})")
    print(f"  {' '.join(scratch_tokens)}")
    
    # 압축률(효율성) 비교
    if base_len > 0:
        diff = base_len - scratch_len
        if diff > 0:
            print(f"\n💡 Scratch_v2가 Baseline보다 {diff}개의 토큰을 더 적게 사용했습니다.")
        elif diff < 0:
            print(f"\n💡 Baseline이 Scratch_v2보다 {abs(diff)}개의 토큰을 더 적게 사용했습니다.")
        else:
            print("\n💡 두 토크나이저의 토큰 사용량이 동일합니다.")
    print("=" * 60)

비교용 토크나이저 로딩 중...
토크나이저 로드 완료.

토크나이저 분절 비교 테스트 (종료하려면 'q' 또는 'quit' 입력)



분절할 텍스트를 입력하세요:  정맥주사



[입력 문장] 정맥주사
------------------------------------------------------------
▶ Baseline (토큰 수: 3)
  정맥 ##주 ##사
------------------------------------------------------------
▶ Scratch_v2 (토큰 수: 1)
  정맥주사

💡 Scratch_v2가 Baseline보다 2개의 토큰을 더 적게 사용했습니다.



분절할 텍스트를 입력하세요:  정맥



[입력 문장] 정맥
------------------------------------------------------------
▶ Baseline (토큰 수: 1)
  정맥
------------------------------------------------------------
▶ Scratch_v2 (토큰 수: 1)
  정맥

💡 두 토크나이저의 토큰 사용량이 동일합니다.


KeyboardInterrupt: Interrupted by user